# RUM Quickstart (synthetic)

A self-contained, top-to-bottom demo of the `RUMModel` front-end on a **synthetic fixed-choice fixture** - no EUROMOD, no France data, no Java. Needs `jax` + `scipy` installed (the lifted solver/SE path); `import dclaborsupply` itself stays light.

## 1. Imports

In [1]:
import tempfile, pathlib
import numpy as np
import dclaborsupply as dcls
from dclaborsupply.spec.parser import EstimationSpec
from dclaborsupply.likelihood._numpy_primitives import PrecomputedDataSingles
print('dclaborsupply', dcls.__version__)

dclaborsupply 0.1.0


## 2. A synthetic fixed-choice spec + precomputed data

A tiny Box-Cox spec. The couples parameters and the wage scale `sigma` are **pinned** via `fixed_params`, so the 8 free *singles* parameters are identified by the male + female singles fixtures. Data is supplied in the engine's precomputed contract (`PrecomputedDataSingles`); raw-dataframe loading is deferred in v0.1.

In [2]:
SPEC_YAML = """specification:
  name: synth_joint
  wage_spec: "vw"
utility:
  functional_form: box_cox
  consumption:
    coefficient: beta_c
    box_cox_exponent: theta_c
  leisure:
    intercept: beta_l0
    box_cox_exponent: theta_l
wage_opportunity:
  specification: "log_normal"
  variance:
    parameter: "sigma"
initial_values:
  beta_l0_sm: 0.8
  beta_c_sm: 0.8
  theta_l_sm: 0.4
  theta_c_sm: 0.4
  beta_l0_sf: 0.8
  beta_c_sf: 0.8
  theta_l_sf: 0.4
  theta_c_sf: 0.4
  beta_l0_m: 1.0
  theta_l_m: 0.5
  beta_l0_f: 1.0
  theta_l_f: 0.5
  beta_c: 1.0
  theta_c: 0.5
  sigma: 0.5
fixed_params:
  beta_l0_m: 1.0
  theta_l_m: 0.5
  beta_l0_f: 1.0
  theta_l_f: 0.5
  beta_c: 1.0
  theta_c: 0.5
  sigma: 0.5
"""

_p = pathlib.Path(tempfile.mkdtemp()) / 'synth_spec.yaml'
_p.write_text(SPEC_YAML, encoding='utf-8')
spec = EstimationSpec.from_yaml(_p)

def make_singles(seed, is_male, n_groups=600, n_alts=6):
    """Small precomputed singles fixture: positive consumption/leisure/log_wage,
    all shifters zero, non-working baseline. No dataframe, no EUROMOD/France data."""
    rng = np.random.default_rng(seed)
    n = n_groups * n_alts
    z = lambda: np.zeros(n)
    c = 0.5 + 4.5 * rng.random(n)
    l = 0.5 + 4.5 * rng.random(n)
    st = np.arange(0, n, n_alts)
    return PrecomputedDataSingles(
        consumption=c, leisure=l, log_c=np.log(c), log_l=np.log(l),
        age_norm=z(), age_norm2=z(), n_children=z(), educL=z(), educM=z(), educH=z(),
        working=z(), working_pt1=z(), working_pt2=z(), working_ft=z(), working_lh=z(), gsur=z(),
        female=z(), in_couple=z(), drgn1=z(),
        reg2=z(), reg3=z(), reg4=z(), reg5=z(), reg6=z(), reg7=z(), reg8=z(),
        drgur=z(), drgmd=z(), drgru=z(),
        year_2015_indicator=z(), year_2017_indicator=z(),
        log_wage=1.0 + rng.random(n), pexp_years=None, pexp_years2=None,
        loc4=None, loc4_1=None, loc4_2=None, loc4_3=None, loc4_4=None,
        prior=1.0 + rng.random(n), c_scale=1.0, l_scale=1.0,
        group_ids=np.arange(n_groups), group_starts=st, group_ends=st + n_alts,
        n_groups=n_groups, n_obs=n,
        actual_choice=z(), cluster_ids=np.arange(n_groups), is_male=is_male)

sm = make_singles(1, is_male=True)
sf = make_singles(2, is_male=False)
print(f'free params ({len(spec.all_param_names)}):', spec.all_param_names)
print('pinned (couples + sigma):', list(spec.fixed_params))

free params (8): ['beta_l0_sm', 'beta_c_sm', 'theta_l_sm', 'theta_c_sm', 'beta_l0_sf', 'beta_c_sf', 'theta_l_sf', 'theta_c_sf']
pinned (couples + sigma): ['beta_l0_m', 'theta_l_m', 'beta_l0_f', 'theta_l_f', 'beta_c', 'theta_c', 'sigma']


## 3. Fit the RUM model

`RUMModel.fit` sums the lifted JAX per-group builders into one objective and optimizes with the lifted SciPy L-BFGS-B solver. No engine math runs in the front end.

In [3]:
model = dcls.RUMModel.from_spec(spec)
result = model.fit((sm, sf, None), compute_se=True)
result.convergence

Hessian is ill-conditioned (cond=1.51e+16)


{'success': True,
 'status': 0,
 'message': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH',
 'n_iter': 252,
 'neg_ll': 2161.896182522086,
 'max_abs_grad': 8.45048192050308}

## 4. Summary

In [4]:
result.summary()

{'model': 'RUM',
 'n_free': 8,
 'n_fixed': 7,
 'neg_ll': 2161.896182522086,
 'converged': True,
 'status': 0,
 'hessian_min_eig': -0.037257238767442205,
 'blocks': {'preference': 8,
  'hours': 0,
  'wage': 0,
  'market': 0,
  'occupation': 0}}

## 5. Parameter blocks

Free parameters partitioned into the five blocks (each parameter assigned exactly once).

In [5]:
result.blocks

{'preference': {'beta_l0_sm': -0.0034833724649631324,
  'beta_c_sm': -1.8780756691782862e-07,
  'theta_l_sm': 3.015653007287228,
  'theta_c_sm': 10.262385504829743,
  'beta_l0_sf': 0.013308067835324115,
  'beta_c_sf': 3.713912568458492e-05,
  'theta_l_sf': 1.7367277333570068,
  'theta_c_sf': 6.4871784173082085},
 'hours': {},
 'wage': {},
 'market': {},
 'occupation': {}}

## 6. Standard errors (exact-JAX Hessian)

In [6]:
result.se('hessian')

{'beta_l0_sm': 0.0037579832844541377,
 'beta_c_sm': 1.5787916072388077e-07,
 'theta_l_sm': 1.5639636240987294e-05,
 'theta_c_sm': 1.7809219777007633e-08,
 'beta_l0_sf': 1.1136727694488596e-09,
 'beta_c_sf': 3.3840239954658744e-05,
 'theta_l_sf': 1.7148903255127662e-11,
 'theta_c_sf': 1.826429700830509e-09}

---
That is the whole v0.1 RUM path: spec -> precomputed data -> `fit` -> `summary` / `blocks` / `se`. Everything dispatches to the lifted core.